<a href="https://colab.research.google.com/github/rund-omari/Data.Eng3/blob/main/lecture/nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
words = ["studies", "running", "happily",'people','men']

print([stemmer.stem(w) for w in words])
# ['studi', 'run', 'happili']

['studi', 'run', 'happili', 'peopl', 'men']


In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)  # first use
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize("studies", pos="v"))
print(lemmatizer.lemmatize("better", pos="a"))
print(lemmatizer.lemmatize("men", pos="n"))
print(lemmatizer.lemmatize("people"))

# study
# good

study
good
men
people


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

reviews = ["good product", "not good product"]

cv = CountVectorizer(ngram_range=(1, 2))
X = cv.fit_transform(reviews)

print(X)
print(cv.get_feature_names_out())
print(X.toarray())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 8 stored elements and shape (2, 5)>
  Coords	Values
  (0, 0)	1
  (0, 4)	1
  (0, 1)	1
  (1, 0)	1
  (1, 4)	1
  (1, 1)	1
  (1, 2)	1
  (1, 3)	1
['good' 'good product' 'not' 'not good' 'product']
[[1 1 0 0 1]
 [1 1 1 1 1]]


In [ ]:
import pandas as pd

In [ ]:
bow_df = pd.DataFrame(X.toarray(),
                      columns=cv.get_feature_names_out())
display(bow_df)

,good,good product,not,not good,product
0,1,1,0,0,1
1,1,1,1,1,1


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
reviews = ["the product is good", "the product is not good",
           "the delivery is late", "the support is good"]


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1,2) ,smooth_idf= True , norm='l2')
X = tfidf.fit_transform(reviews)
weighted_df = pd.DataFrame(X.toarray(),
                           columns=tfidf.get_feature_names_out())
display(weighted_df.round(3))

,delivery,delivery is,good,is,is good,is late,is not,late,not,not good,product,product is,support,support is,the,the delivery,the product,the support
0,0.000,0.000,0.344,0.281,0.425,0.000,0.000,0.000,0.000,0.000,0.425,0.425,0.000,0.000,0.281,0.000,0.425,0.000
1,0.000,0.000,0.265,0.216,0.000,0.000,0.415,0.000,0.415,0.415,0.327,0.327,0.000,0.000,0.216,0.000,0.327,0.000
2,0.425,0.425,0.000,0.222,0.000,0.425,0.000,0.425,0.000,0.000,0.000,0.000,0.000,0.000,0.222,0.425,0.000,0.000
3,0.000,0.000,0.298,0.244,0.369,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.468,0.468,0.244,0.000,0.000,0.468


In [ ]:
import pandas as pd
import re
import html
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Keep negation; do not remove it automatically.
def clean_text(value):
    if pd.isna(value) or not str(value).strip():
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    return " ".join(text.lower().split())

In [ ]:
def clean_series(series):
    return pd.Series(series).apply(clean_text)

In [ ]:
df = pd.read_csv("/content/SDK_NLP_Customer_Reviews_Practice.csv")
df

,review_id,review_date,product_category,channel,rating,review_text
0,REV-0001,2026-01-04,Beauty,Marketplace,1,Never ordering again. Support was unhelpful.
1,REV-0002,2026-01-06,Accessories,Mobile App,2,The color is different from the photo.
2,REV-0003,2026-01-08,Electronics,Website,2,I need to contact support@example.com today.
3,REV-0004,2026-01-10,Home,Marketplace,5,This product is better than expected.
4,REV-0005,2026-01-12,Fashion,Mobile App,4,Order arrived in Amman within 2 days.
...,...,...,...,...,...,...
115,REV-0116,2026-08-22,Beauty,Mobile App,3,This is okay... nothing special.
116,REV-0117,2026-08-24,Accessories,Website,3,Product quality is good; shipping was SLOW.
117,REV-0118,2026-08-26,Electronics,Marketplace,5,Very good value for 25 JOD.
118,REV-0119,2026-08-28,Home,Mobile App,1,"No, I would not recommend it."


In [ ]:
X = df["review_text"].fillna("")
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

In [ ]:
pipeline = Pipeline([
    ("clean", FunctionTransformer(clean_series, validate=False)),
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=500))
])


In [ ]:
X_train_tfidf = pipeline.fit_transform(X_train)
X_test_tfidf = pipeline.transform(X_test)

print("Train:", X_train_tfidf.shape)
print("Test:", X_test_tfidf.shape)
print(pipeline.named_steps["tfidf"].get_feature_names_out()[:15])

Train: (96, 330)
Test: (24, 330)
['10' '10 10' '12' '12 days' '25' '25 jod' 'after' 'after just' 'again'
 'again support' 'all' 'all want' 'amazing' 'amazing quality' 'amman']


In [ ]:
pd.Series([['text','hi','waedgfgb']]).apply(lambda xs: [len(w) for w in xs])

,0
0,"[4, 2, 8]"


In [ ]:
import re
import pandas as pd

def extract_text_features(series):
    s = pd.Series(series).fillna("").astype(str)
    words = s.str.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?")
    lengths = words.apply(lambda xs: [len(w) for w in xs])

    return pd.DataFrame({
        "char_count": s.str.len(),
        "word_count": words.str.len(),
        "exclamation_count": s.str.count(r"!"),
        "has_url": s.str.contains(
            r"https?://\S+|www\.\S+", regex=True, case=False
        ).astype(int),
        "has_negation": s.str.contains(
            r"\b(?:not|no|never)\b|n't\b",
            regex=True, case=False
        ).astype(int),
        "avg_word_length": lengths.apply(
            lambda xs: sum(xs) / len(xs) if xs else 0.0
        )
    }, index=s.index)

stats = extract_text_features(df["review_text"])
print(stats.head())

   char_count  word_count  exclamation_count  has_url  has_negation  \
0          44           6                  0        0             1   
1          38           7                  0        0             0   
2          44           8                  0        0             0   
3          37           6                  0        0             0   
4          37           6                  0        0             0   

   avg_word_length  
0         6.166667  
1         4.428571  
2         4.500000  
3         5.166667  
4         4.833333  


In [ ]:
import numpy as np

reviews = df["review_text"].astype("string")

missing_or_blank = (
    reviews.isna()
    | reviews.str.strip().eq("").fillna(False)
)

cleaned = reviews.fillna("").map(clean_text)
empty_after_cleaning = cleaned.str.strip().eq("")

print("Missing / blank:", missing_or_blank.sum())
print("Empty after cleaning:", empty_after_cleaning.sum())
print("Exact duplicates:", reviews.duplicated().sum())

# After fitting the pipeline:
assert X_train_tfidf.shape[0] == len(X_train)
assert X_test_tfidf.shape[0] == len(X_test)
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]
assert np.isfinite(X_train_tfidf.data).all()

Missing / blank: 8
Empty after cleaning: 8
Exact duplicates: 76
